In [1]:
#Imports
#General Python Tools
import os
import json
import random
from pathlib import Path

#Data Handling & Mathematics
import numpy as np
import pandas as pd

#Image & Video Processing
import cv2
from PIL import Image

#Graphs & Visuals
import matplotlib.pyplot as plt

#Analyzing Ripple Motions & Signals
from scipy import signal 
from scipy.ndimage import gaussian_filter

#Machine Learning
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.multioutput import MultiOutputClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    multilabel_confusion_matrix
)

#Saving & Loading Trained Models
import joblib

#Progress Tracking
from tqdm.auto import tqdm 


In [2]:
#Environment Setup
#Sets a consistent random seed so experiments are reproducible
SEED = 53
random.seed(SEED)
np.random.seed(SEED)
cv2.setRNGSeed(SEED)

#Identifies Kaggle's input & output directories
INPUT_DIR = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working/liquidlens_outputs")
#Creates output folder if it doesn't already exist
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Liquidlens environment successfully intialized!")
print(f"Input directory:{INPUT_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Random seed: {SEED}")

Liquidlens environment successfully intialized!
Input directory:/kaggle/input
Output directory: /kaggle/working/liquidlens_outputs
Random seed: 53


In [3]:
#Dataset Loading & Organization
#Defining Supported File Types
IMAGE_EXTENSIONS = {
    ".jpeg",".jpg",".png",".bmp",".tif",".tiff"
}
VIDEO_EXTENSIONS = {
    ".mp4",".avi",".mov",".mkv",".wmv"
}
#Dataset Files
def discover_dataset_files(root_dir):
    #Store Information about every discovered media file
    files = [] 
    # Search through input directory and its subfolders
    for file_path in root_dir.rglob("*"):
        #Ignore Folders & Process actual files
        if not file_path.isfile():
            continue
        #Identify the file
        extension = file_path.suffix.lower()
        #Determine whether the file is an image or video
        if extension in IMAGE_EXTENSIONS:
            media_type = "image"
        elif extension in VIDEO_EXTENSIONS:
            media_type = "video"
        else:
            continue

        #Save the file's information
        files.append({
            "file_name": file_path.name,
            "file_path": str(file_path),
            "extension": extension,
            "media_type": media_type,
            "parent_folder": file_path.parent.name
        })
    #Converts the collected information into a DataFrame
    return pd.DataFrame(
        files,columns=[
            "file_name",
            "file_path",
            "extension",
            "media_type",
            "parent_folder"
        ]
    )

#Load Dataset Inventory
media_df = discover_dataset_files(INPUT_DIR)
#Locate sample metadata
metadata_files = list(INPUT_DIR.rglob("metadata.csv"))
#Display Info
print("LiquidLens Dataset Scan")
print("-"*40)
print(f"Total images found: {(media_df['media_type'] == 'image').sum()}")
print(f"Total videos found: {(media_df['media_type'] == 'video').sum()}")
print(f"Total media files: {len(media_df)}")
print(f"Metadata files found: {len(metadata_files)}")

if not media_df.empty:
    print("\nDataset File Inventory:")
    display(media_df.head(10))
else:
    print("\nNo Image or Video Files Found Yet.")
    print("Attach your dataset through Kaggle's Add Input Option.")
if metadata_files:
    print("\nMetadata file locations:")
    for metadata_path in metadata_files:
        print(metadata_path)

else:
    print("\nNo metadata.csv file found yet.")
    print("This is expected if the dataset has not been uploaded.")
    


LiquidLens Dataset Scan
----------------------------------------
Total images found: 0
Total videos found: 0
Total media files: 0
Metadata files found: 0

No Image or Video Files Found Yet.
Attach your dataset through Kaggle's Add Input Option.

No metadata.csv file found yet.
This is expected if the dataset has not been uploaded.


In [4]:
#Image Processing
def load_image(image_path):
    #Reads the image from its file path
    image = cv2.imread(str(image_path))
    #Checks whether the image was loaded successfully
    if image is None:
        raise ValueError(f"Unable to load image: {image_path}")
    #OpenCV loads images in BGR format.
    #Convert BGR into RGB for accurate color display
    image_rgb = cv2.cvtColor(image,cv2.COLOR_BGR2RGB)

    #Return the original color info in RGB format
    return image_rgb

#Finding available image files
image_files = media_df[
    media_df["media_type"] == "image"
]["file_path"].tolist()

#Load/Inspect an Image
if image_files:
    #load the first availale image
    sample_image = load_image(image_files[0])
    #Retrieve image dimensions
    height,width,channels = sample_image.shape
    print("Image Inspection")
    print("-"* 40)
    print(f"Image width: {width} pixels")
    print(f"Image height: {height} pixels")
    print(f"Color channels: {channels}")
    print(f"Image data type: {sample_image.dtype}")
    
    #Display Image without changing original data
    plt.figure(figsize=(8,6))
    plt.imshow(sample_image)
    plt.title("LiquidLens - Original Beverage Image")
    plt.axis ("off")
    plt.show()

else:
    print("No images are currently available!")
    print("Uploas your image dataset in order to test this function!")

No images are currently available!
Uploas your image dataset in order to test this function!
